# 23 — Demographics vs. Homeless Rate (City/CoC)

Correlations between homeless rate and racial/ethnic composition at CoC level.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_city_data.csv')
df_demo = df.dropna(subset=['homeless_rate_per_10k', 'pct_black']) if 'pct_black' in df.columns else pd.DataFrame()
print(f'CoCs with demographics: {len(df_demo)}')

CoCs with demographics: 0


In [2]:
if len(df_demo) > 5:
    slope, intercept, r, p, se = stats.linregress(df_demo['pct_black'], df_demo['homeless_rate_per_10k'])
    r2 = r ** 2
    x_range = [df_demo['pct_black'].min(), df_demo['pct_black'].max()]
    y_range = [slope * x + intercept for x in x_range]
    fig = px.scatter(
        df_demo, x='pct_black', y='homeless_rate_per_10k', text='city_name',
        color='homeless_rate_per_10k', color_continuous_scale='Reds',
        hover_name='coc_name',
        title=f'% Black Population vs. Homeless Rate (City/CoC)<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f}</sup>',
        labels={'pct_black': '% Black (non-Hispanic)', 'homeless_rate_per_10k': 'Rate per 10k'},
    )
    fig.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='darkred', dash='dash')))
    fig.update_traces(textposition='top center', selector=dict(mode='markers+text'))
    fig.show()
else:
    # Show unsheltered % breakdown by state as demographic proxy
    state_avg = df.groupby('state_postal').agg(
        avg_unsheltered_pct=('unsheltered_pct', 'mean'),
        avg_rate=('homeless_rate_per_10k', 'mean'),
        total=('total_homeless', 'sum')
    ).reset_index()
    fig = px.scatter(
        state_avg, x='avg_unsheltered_pct', y='avg_rate', text='state_postal',
        size='total', color='avg_rate', color_continuous_scale='Reds',
        title='Avg Unsheltered % vs. Avg Homeless Rate by State (demographic proxy)',
        labels={'avg_unsheltered_pct': 'Avg Unsheltered %', 'avg_rate': 'Avg Rate per 10k'},
    )
    fig.update_traces(textposition='top center', selector=dict(mode='markers+text'))
    fig.show()